### **Orchestrator-Worker Coding Agent**

User Request

      ↓
Orchestrator

      ↓
Generate Tasks

      ↓
      
Send()

      ↓

Workers (Parallel)

      ↓
Generate Code

      ↓
Synthesizer

      ↓
Final Output

In [4]:
! pip install langchain_core langchain-nvidia-ai-endpoints langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.2 MB/s eta 0:00:00
  Attempting uninstall: aiohttp
    Found existing installation: aiohttp 3.14.1
    Uninstalling aiohttp-3.14.1:
      Successfully uninstalled aiohttp-3.14.1


In [3]:
from typing import Annotated, List
import operator
from langgraph.types import Send
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


In [17]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import os

llm = ChatNVIDIA(
    model=os.getenv("LLM_MODEL"),
    api_key=os.getenv('API_KEY'),
    temperature=0
)

In [7]:
class State(TypedDict):
    project: str
    tasks: list[str]
    generated_code: Annotated[list[str], operator.add]
    final_output: str

In [8]:
class WorkerState(TypedDict):
    task: str
    generated_code: Annotated[list[str], operator.add]


In [9]:
def orchestrator(state: State):
    """
    Break project into tasks
    """

    project = state["project"]

    prompt = f"""
    You are a software architect.

    Break this project into coding tasks.

    Project:
    {project}

    Return only a Python list.

    Example:
    ["Entity","Repository","Service","Controller"]
    """

    response = llm.invoke(prompt)

    try:
        tasks = eval(response.content)
    except:
        tasks = [
            "Entity",
            "Repository",
            "Service",
            "Controller"
        ]

    return {
        "tasks": tasks
    }


In [10]:
def assign_workers(state: State):

    return [
        Send(
            "code_generator",
            {
                "task": task
            }
        )
        for task in state["tasks"]
    ]


In [11]:
def code_generator(state: WorkerState):

    task = state["task"]

    prompt = f"""
    Generate Spring Boot code for:

    {task}

    Return only code.
    """

    response = llm.invoke(prompt)

    return {
        "generated_code": [
            f"""
====================
{task}
====================

{response.content}
"""
        ]
    }


In [12]:
def synthesizer(state: State):

    final_result = "\n\n".join(
        state["generated_code"]
    )

    return {
        "final_output": final_result
    }


In [13]:
builder = StateGraph(State)

builder.add_node(
    "orchestrator",
    orchestrator
)

builder.add_node(
    "code_generator",
    code_generator
)

builder.add_node(
    "synthesizer",
    synthesizer
)

builder.add_edge(
    START,
    "orchestrator"
)

builder.add_conditional_edges(
    "orchestrator",
    assign_workers,
    ["code_generator"]
)

builder.add_edge(
    "code_generator",
    "synthesizer"
)

builder.add_edge(
    "synthesizer",
    END
)

graph = builder.compile()

In [18]:
result = graph.invoke(
    {
        "project": "Build a Notes CRUD API"
    }
)

print(result["final_output"])


Setup project structure





Define Note entity

```java
package com.example.demo.model;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
@Table(name = "notes")
public class Note {

    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;

    private String title;

    @Column(columnDefinition = "TEXT")
    private String content;

    @Column(name = "created_at", updatable = false)
    private LocalDateTime createdAt;

    @Column(name = "updated_at")
    private LocalDateTime updatedAt;

    public Note() {
    }

    public Note(String title, String content) {
        this.title = title;
        this.content = content;
    }

    @PrePersist
    protected void onCreate() {
        this.createdAt = LocalDateTime.now();
        this.updatedAt = this.createdAt;
    }

    @PreUpdate
    protected void onUpdate() {
        this.updatedAt = LocalDateTime.now();
    }

    public Long getId() {
        return id;
    }

    public void 